# Setup

In [1]:
import sys
sys.path.append('../../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
from pandas import DataFrame
from processor.llm.interface.model_interface import ModelInterface
from processor.llm.interface.model_factory import get_model
from processor.utils import format_schema_with_samples
from tqdm import tqdm
from processor.types.message import Message

In [2]:
ckp = '../llm/weight/qwen25-7b'
interface = get_model(ckp)
model: ModelInterface = interface(ckp)
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [32]:
base_table_producer_prompts = {
    "tables_selector": """You are an experienced data scientist. You are given:
- A table, represented by its schema, a description of what it contains, and some sample rows. The pipe character (`|`) is used as the separator for both columns and row values.
- A target schema that needs to be constructed using one or more of the available tables.

Your task is to determine whether this table is **relevant** for constructing the target schema — either fully or partially. A table is considered relevant if it provides **any** useful information toward fulfilling the target schema, such as:
- Matching any of the target columns exactly,
- Providing a column that can be transformed into a target column,
- Contributing auxiliary information (e.g., geographic clues from `city` or `address` that help construct `Is in Bay Area`).

Err on the side of inclusion: if you think even **one** column might help, mark the table as **relevant**.

End your reasoning with the following exact format, to ease parsing:

Relevant: yes/no
""",
    "row_extender": """You are an experienced data scientist. You are given:
- A list of tables, each with its schema, description, and a few sample rows.
- The pipe character (`|`) is used to separate columns and row values.

Your task is to find sets of tables that can be combined via **row extension** (i.e., stacking the rows together). Each group must:
- Represent the same type of entity (e.g., coffee shops),
- Have compatible schemas (even if column names differ),
- Have columns that can be aligned or initialized to `null` if missing in some tables.

Produce a **merge plan** in JSON format. For each group of mergeable tables, specify:
- `"Tables"`: a list of table IDs (e.g., ["Table_0", "Table_2"]),
- `"Schema"`: the unified list of columns after merge,
- `"Mappings"`: a dictionary that maps each table to how its columns align with the unified schema. If a column is missing, you may leave it out or assume it will be filled with nulls.

Output your response as a JSON list of such merge plans, with no extra commentary.

Example output:

```json
[
  {
    "Tables": ["Table_0", "Table_1"],
    "Schema": ["Restaurant ID", "Name", "Rating", "City"],
    "Mappings": {
      "Table_0": {"ID": "Restaurant ID", "NAME": "Name", "RATING": "Rating"},
      "Table_1": {"restaurant_id": "Restaurant ID", "name": "Name", "rating": "Rating", "city": "City"}
    }
  }
]
""",
}

# Base Table Producer

In [41]:
def __select_tables(available_tables: list[DataFrame], tables_descs: list[str], target_schema: list[str]):
    relevant_tables: list[DataFrame] = []
    for table_idx, table in enumerate(available_tables):
        msg: list[Message] = [
            {'role': 'system', 'content': base_table_producer_prompts['tables_selector']},
            {'role': 'user', 'content': f"- Table: {format_schema_with_samples(table)}\n\n- Target schema: {target_schema}\n\n- Description: {tables_descs[table_idx]}" },
        ]
        table_relevancy_output = model.chat(msg)
        print(f"=> table_relevancy_output: {table_relevancy_output}")
        table_relevance = table_relevancy_output.split('Relevant: ')[-1].lower().strip()
        if table_relevance.startswith('yes'):
            print(f"==> Yes, this table is relevant!")
            relevant_tables.append(True)
    return relevant_tables

In [42]:
zomato = pd.read_csv("../../../data_src/zomato.csv")
zomato.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "Customer_Rating",
    "TELEPHONE_NUMBER",
    "NUMBER_OF_REVIEWS",
    "Full Address",
]
yelp = pd.read_csv("../../../data_src/yelp.csv")
yelp.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "RESTAURANT_RATING",
    "PHONE_NUMBER_FORMATTED",
    "REVIEWS",
    "LOCATION",
]

available_tables = [zomato, yelp]
tables_descs = [
    'This table likely represents restaurant listings including ratings, contact information, and addresses.',
    'This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.',
]
target_schema = ['Restaurant ID', 'Restaurant Name', 'Ratings', 'Is in Bay Area']
relevant_tables = __select_tables(available_tables, tables_descs, target_schema)
relevant_tables  # Yes, Yes!

=> table_relevancy_output: This table contains columns that match or can be transformed to match the target schema:

1. `RESTAURANT_ID` can be mapped directly to `Restaurant ID`.
2. `RESTAURANT_NAME` can be mapped directly to `Restaurant Name`.
3. `Customer_Rating` can be mapped to `Ratings`.
4. While `Full Address` does not directly map to `Is in Bay Area`, it provides the necessary geographic information to determine if a restaurant is in the Bay Area.

Given these mappings, the table provides useful information towards constructing the target schema.

Relevant: yes
==> Yes, this table is relevant!
=> table_relevancy_output: The table contains columns that match or can be transformed to match the target schema:

- `RESTAURANT_ID` matches 'Restaurant ID'
- `RESTAURANT_NAME` matches 'Restaurant Name'
- `RESTAURANT_RATING` can be transformed to 'Ratings' (by removing the decimal part)
- The `LOCATION` column can provide auxiliary information to determine if a restaurant is in the Bay Ar

[True, True]

# Row Extension

In [43]:
def __extend_tables(available_tables: list[DataFrame], tables_descs: list[str]):
    relevant_tables: list[DataFrame] = []
    available_tables_formatted = ""
    for table_idx, table in enumerate(available_tables):
        available_tables_formatted += f"- Table {table_idx} ({tables_descs[table_idx]}):\n```{format_schema_with_samples(table)}```\n\n"
    available_tables_formatted = available_tables_formatted.strip()
    print(f"=> available_tables_formatted: {available_tables_formatted}")

    msg: list[Message] = [
        {'role': 'system', 'content': base_table_producer_prompts['row_extender']},
        {'role': 'user', 'content': available_tables_formatted},
    ]
    extend_operations = model.chat(msg)
    print(f"=> extend_operations: {extend_operations}")
    return extend_operations

In [44]:
zomato = pd.read_csv("../../../data_src/zomato.csv")
zomato.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "Customer_Rating",
    "TELEPHONE_NUMBER",
    "NUMBER_OF_REVIEWS",
    "Full Address",
]
yelp = pd.read_csv("../../../data_src/yelp.csv")
yelp.columns = [
    "RESTAURANT_ID",
    "RESTAURANT_NAME",
    "RESTAURANT_RATING",
    "PHONE_NUMBER_FORMATTED",
    "REVIEWS",
    "LOCATION",
]

available_tables = [zomato, yelp]
tables_descs = [
    "This table likely represents restaurant listings including ratings, contact information, and addresses.",
    "This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.",
]
extend_operations = __extend_tables(available_tables, tables_descs)
extend_operations  # Yes, Yes!

=> available_tables_formatted: - Table 0 (This table likely represents restaurant listings including ratings, contact information, and addresses.):
```col: RESTAURANT_ID | RESTAURANT_NAME | Customer_Rating | TELEPHONE_NUMBER | NUMBER_OF_REVIEWS | Full Address
sample row 1: 1450000000291 | Big & Little's  | 3.8 | (773) 857-6677 | 36 | 1034 W. Belmont Avenue, Chicago, IL
sample row 2: 1450000002328 | Salonica  | 3.8 | (773) 752-3899 | 130 | 1440 E. 57th Street, Chicago, IL
sample row 3: 1450000001462 | La Brioche  | 3.6 | (608) 233-3388 | 257 | 2862 University Ave, Madison, WI```

- Table 1 (This table likely represents information about restaurants or cafes, including their name, rating, contact number, number of reviews, and address.):
```col: RESTAURANT_ID | RESTAURANT_NAME | RESTAURANT_RATING | PHONE_NUMBER_FORMATTED | REVIEWS | LOCATION
sample row 1: 1445980005373 | The Village Idiot  | 3.5 | (323) 655-3331 | 958 | 7383 Melrose Ave, Los Angeles, CA 90046
sample row 2: 1445980005301 

=> extend_operations: ```json
[
  {
    "Tables": ["Table_0", "Table_1"],
    "Schema": ["RESTAURANT_ID", "RESTAURANT_NAME", "Customer_Rating", "PHONE_NUMBER_FORMATTED", "REVIEWS", "LOCATION"],
    "Mappings": {
      "Table_0": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "Customer_Rating": "Customer_Rating", "TELEPHONE_NUMBER": "PHONE_NUMBER_FORMATTED", "NUMBER_OF_REVIEWS": "REVIEWS", "Full Address": "LOCATION"},
      "Table_1": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "RESTAURANT_RATING": "Customer_Rating", "PHONE_NUMBER_FORMATTED": "PHONE_NUMBER_FORMATTED", "REVIEWS": "REVIEWS", "LOCATION": "LOCATION"}
    }
  }
]
```


'```json\n[\n  {\n    "Tables": ["Table_0", "Table_1"],\n    "Schema": ["RESTAURANT_ID", "RESTAURANT_NAME", "Customer_Rating", "PHONE_NUMBER_FORMATTED", "REVIEWS", "LOCATION"],\n    "Mappings": {\n      "Table_0": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "Customer_Rating": "Customer_Rating", "TELEPHONE_NUMBER": "PHONE_NUMBER_FORMATTED", "NUMBER_OF_REVIEWS": "REVIEWS", "Full Address": "LOCATION"},\n      "Table_1": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "RESTAURANT_RATING": "Customer_Rating", "PHONE_NUMBER_FORMATTED": "PHONE_NUMBER_FORMATTED", "REVIEWS": "REVIEWS", "LOCATION": "LOCATION"}\n    }\n  }\n]\n```'

In [59]:
# Parse the operation
import json
operation_json = '```json\n[\n  {\n    "Tables": ["Table_0", "Table_1"],\n    "Schema": ["RESTAURANT_ID", "RESTAURANT_NAME", "Customer_Rating", "PHONE_NUMBER_FORMATTED", "REVIEWS", "LOCATION"],\n    "Mappings": {\n      "Table_0": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "Customer_Rating": "Customer_Rating", "TELEPHONE_NUMBER": "PHONE_NUMBER_FORMATTED", "NUMBER_OF_REVIEWS": "REVIEWS", "Full Address": "LOCATION"},\n      "Table_1": {"RESTAURANT_ID": "RESTAURANT_ID", "RESTAURANT_NAME": "RESTAURANT_NAME", "RESTAURANT_RATING": "Customer_Rating", "PHONE_NUMBER_FORMATTED": "PHONE_NUMBER_FORMATTED", "REVIEWS": "REVIEWS", "LOCATION": "LOCATION"}\n    }\n  }\n]\n```'
if operation_json.startswith('```'):
    operation_json = operation_json[3:]
if operation_json.endswith('```'):
    operation_json = operation_json[:-3]
if operation_json.startswith('json'):
    operation_json = operation_json[4:]
operations = json.loads(operation_json)

In [60]:
extended_dfs = []
tables = {
    "Table_0": zomato,
    "Table_1": yelp,
}
last_idx = 1
for operation in operations:
    normalized_tables = []
    for table_name in operation["Tables"]:
        df = tables[table_name]
        mapping = operation["Mappings"][table_name]

        renamed_df = df.rename(columns=mapping)
        schema = operation["Schema"]
        for col in schema:
            if col not in renamed_df.columns:
                renamed_df[col] = None

        renamed_df = renamed_df[schema]
        normalized_tables.append(renamed_df)
    extended_df = pd.concat(normalized_tables, ignore_index=True)
    extended_dfs.append(extended_df)
for extended_df in extended_dfs:
    tables[f'Table_{last_idx+1}'] = extended_df
    last_idx += 1
tables['Table_2'].to_csv('Yelp-Zomato.csv', index=False)

# Join Operations